# Connect and Login

In [1]:
import win32com.client

# Don't use admin credential unless needed
# connection_name = 'JGO_CO1SQLWPV22'
# user_name = 'admin-robot'
# password='098765'

# Default credentials - use windows authentication
connection_name = ''
user_name = ''
password = ''

ResQApp = win32com.client.Dispatch("ResQ3Automation.ResQApplication")
ResQApp.ConnectByName('JGO_CO1SQLWPV22', user_name, password) # use window authentication

In [27]:
ResQApp.Disconnect() # Release license after work complete

# Select project and reserving class path

In [18]:
ProjectName = 'NJ_Annual_Prod_202605_Fake'
# ProjectName = 'NJ_Annual_Prod_2026 Q3-Aug'
# ProjectName = 'NJ_Annual_Prod_2026 Q2-May'
# Path = r'PRNJ - PA\PA\All States\Direct Group\CMPxCAT'
# Path = r"PRNJ - PA\PA\NY\Direct Group\MP+PIP"
# Path = r"HPPREF\HO+DF\NJ\Legacy\HOL"
# Path = r'PRNJ - PA\PA\All States\Direct Group\COL'
Path = r"PRNJ - PA\PA\NJ\Direct Group\MP+PIP"  # Has tail factors in DFM

project = ResQApp.Projects().Item(ProjectName)
reserving_class = project.ReservingClasses().Item(Path)

# Datasets

In [ ]:
len(reserving_class.Vectors()) + len(reserving_class.Triangles())

## Triangles

In [3]:
TriangleName = 'Gross Loss--Paid  - B&S Settlement Rate Adjustment'
triangle = reserving_class.Triangles().Item(TriangleName)

# DataFormat: 0=Triangle
triangle.Name
triangle.DatasetType.DataFormat

0

In [ ]:
triangle.DatasetType.Name

In [ ]:
from datetime import datetime

triangle.OriginLabel(1) # use the first row label, count the development columns
triangle.DevelopmentCount(OriginDate=datetime(int(triangle.OriginLabel(1)), 12, 31))

In [ ]:
top_n = 3
count = 0

for i in reserving_class.Triangles():
    print([i.Name, i.DatasetType.Name, i.DatasetType.DataFormat, i.User, i.Created, i.Modified])
    count += 1
    if count >= top_n:
        break

## Vectors

In [ ]:
[i.Name for i in reserving_class.Vectors()]

In [ ]:
vector = reserving_class.Vectors().Item('C 91 -  Current Qtr Indicated - Feb 2026')
vector.DatasetType.Category.Name

In [ ]:
VectorName = 'C 41 - BF Reported ex CWOP'
VectorName = 'C 61 Reported - CWOP'
vector = reserving_class.Vectors().Item(VectorName)

# DataFormat: 1=Vector
vector.DatasetType.DataFormat

In [ ]:
vector.Formula

### Read and write vector values


In [ ]:
# ResQ vectors are one-dimensional datasets. Values are addressed by 1-based origin index.
origin_index = 1

vector.Count
vector.OriginLabel(origin_index)
vector.ValuesByIndex(origin_index)


In [ ]:
# Write one vector value back to ResQ. Use with care on a scratch/test project.
# new_value = vector.ValuesByIndex(origin_index)
# vector.SetValuesByIndex(origin_index, new_value)
# vector.Save()


In [ ]:
# ResQ method type codes used by vector.MethodType / OutputVector.MethodType
METHOD_TYPES = {
    0: "None",
    1: "DFM",
    2: "BF",
    3: "CC",
    4: "Result Selection",
}

vector.MethodType
METHOD_TYPES.get(vector.MethodType, vector.MethodType)


In [ ]:
vector.Method.Name 

In [ ]:
vector.MethodType
# MethodType: 0=None, 1=DFM, 2=BF, 3=CC, 4=Result Selection


### Example: adjust the vector input values

In [30]:
RC_PATH = [
    r"PRNJ - PA\PA\NY\Direct Group\BI Total",
    r"PRNJ - PA\PA\NY\Direct Group\MP+PIP",
    r"PRNJ - PA\PA\Penn+CT\Direct Group\BI Total",
    r"PRNJ - PA\PA\Penn+CT\Direct Group\MP+PIP",
    r"PRNJ - PA\PA\All States\Direct Group\PD+UMPD",
    r"PRNJ - PA\PA\All States\Direct Group\COL",
    r"PRNJ - PA\PA\All States\Direct Group\CMPxCAT",
    r"PRNJ - PA\PA\NJ\Direct Group\MP+PIP",
    r"PRNJ - PA\PA\NJ\Direct Group\BIR51+UMBIR51",
    r"PRNJ - PA\PA\NJ\Direct Group\BIx51+UMBIx51",
    r"HPPREF\HO+DF\NJ\Legacy\HOL",
    r"HPPREF\HO+DF\NJ\Legacy\HOPxCAT",
    r"Rider\MC\All States\Direct Group\BI+PIP", 
    r"Rider\MC\All States\Direct Group\PD+UMPD", 
    r"Rider\MC\All States\Direct Group\PhysDxCat",
    r"PRNJ - PA\PA\MA\Direct Group\BI Total",
    r"PRNJ - PA\PA\MA\Direct Group\MP+PIP",
]

In [45]:
p0 = ResQApp.Projects().Item('NJ_Annual_Prod_2026 Q2-May')
project = ResQApp.Projects().Item('NJ_Annual_Prod_202605_Fake')

for reserving_class_path in RC_PATH:
    r0 = p0.GetReservingClass(reserving_class_path)
    reserving_class = project.GetReservingClass(reserving_class_path)
    for vector in reserving_class.Vectors():
        v0 = r0.GetVector(vector.Name)
        if vector.Calculated==False and vector.MethodType==0:
            if 'Remaining' in vector.Name: continue
            print(f"{reserving_class_path}...{vector.Name}")

            display_length = vector.PeriodLength
            vector.PeriodLength = vector.StoredPeriodLength

            for i in range(1, vector.Count+1):
                vector.SetValuesByIndex(i, v0.ValuesByIndex(i)*2.4)

            vector.PeriodLength = display_length
            vector.Save()

PRNJ - PA\PA\NY\Direct Group\BI Total...C 42a - Prior for BF Reported ex CWOP
PRNJ - PA\PA\NY\Direct Group\BI Total...P 06 Net Loss--Expected Net Loss % of Earned Premium 
PRNJ - PA\PA\NY\Direct Group\BI Total...P 04 B-ALAE--Expected % of Ultimate Net Loss - Indicated
PRNJ - PA\PA\NY\Direct Group\BI Total...F 91 - Current Qtr Indicated - Feb 2026
PRNJ - PA\PA\NY\Direct Group\BI Total...F 92 - Current Qtr Selected  - Feb 2026
PRNJ - PA\PA\NY\Direct Group\BI Total...C 92 -  Current Qtr Selected - Feb 2026
PRNJ - PA\PA\NY\Direct Group\BI Total...C 92 -  Current Qtr Selected - Feb 2026
PRNJ - PA\PA\NY\Direct Group\BI Total...C 91 -  Current Qtr Indicated - Feb 2026
PRNJ - PA\PA\NY\Direct Group\BI Total...G 91 - Current Qtr Indicated  - Feb 2026
PRNJ - PA\PA\NY\Direct Group\BI Total...G 91 - Current Qtr Indicated  - Feb 2026
PRNJ - PA\PA\NY\Direct Group\BI Total...G 92 - Current Qtr Selected  - Feb 2026
PRNJ - PA\PA\NY\Direct Group\BI Total...G 92 - Current Qtr Selected  - Feb 2026
PRNJ - P

# DFM method properties

## Read properties

In [3]:
# In ResQ, some instance name and dataset type name have unncessary white space(s) inside/after the name, 
# during the transition, we need to trim those white spaces and use a clean version and standarized version of the names
all_DFMs = list(i.Name for i in reserving_class.DFMMethods())  # always use reserving_class.DFMMethods, not project.DFMMethods
all_DFMs

['C 12 - CWP DFM w/ Selected LDFs ',
 'C 42 - Reported ex CWOP DFM w/ Selected LDFs  ',
 'C 32 - Reported DFM w/ Selected LDFs   ',
 'C 22 - CWOP DFM w/ Selected LDFs  ',
 'C 52 - CWOP/Reported DFM w/ Selected LDFs  ',
 'E1 12 - Salv DFM w/ Selected LDFs  ',
 'D 13 - Paid DFM w/ Selected LDFs',
 'D 18 - BS Paid DFM',
 'D 23 - Incurred DFM w/ Selected LDFs',
 'E1 22 - Salv/Paid Loss DFM w/ Selected LDFs',
 'H 01 - Gross Incurred per Reported ex CWOP DFM w/ Selected LDFs   ',
 'G 12 - ALAE--Paid DFM w/ Selected LDFs  ',
 'G 22 A - ALAE/Gross Paid Loss DFM w/ Selected LDFs',
 'E2 12 - Subr DFM w/ Selected LDFs   ',
 'E2 22 - Subr/Paid Loss DFM w/ Selected LDFs ',
 'F 25 - Incurred DFM Bootstrap',
 'C 43 - DFM for BF Reported ex CWOP']

In [19]:
# Development Factor Method (DFM) ... The api method name is DFMMethods ...
DFM_MethodName = r'C 12 - CWP DFM w/ Selected LDFs '

aDFM = reserving_class.DFMMethods().Item(DFM_MethodName)

org_rng = range(1, aDFM.OriginCount+1)
dev_rng = range(1, aDFM.DevelopmentCount(1)+1)

aDFM.OriginLength
aDFM.DevelopmentLength

aDFM.OriginCount # number of rows
aDFM.DevelopmentCount(1) # number of dev cols (look at the first row)

# aDFM.ExcludedRatios(i, j) 
# return values 0, 1, 2 for triangle cell i, j
# 0=included; 
# 1=excluded; 
# 2=empty cell (no value)

# ratio selection status (pattern)
excluded_ratio_pattern = [[aDFM.ExcludedRatios(i, j) for j in dev_rng] for i in org_rng]

# Other Attributes
try:
    aDFM.SummaryRatioBasis.Name # ratio_basis_dataset
except:
    print('Ratio Basis not Selected')
aDFM.RatioDecimalPlaces
aDFM.SummaryRatioDecimalPlaces
aDFM.InputTriangle.Name
aDFM.OutputVector.Name
aDFM.OutputVector.Modified # returns pywintypes.datetime(yyyy, m, d, h, m, s, ..., tzinfo=TimeZoneInfo('GMT Standard Time', True))

average_formulas = [aDFM.AverageFormula(i) for i in range(1, 15)] # get all average formula names

# capped the max row at 20 since all values after 13 will generally be 'XX: User Entry'
# remove "<index>: " before the actual formula name
# only allow one User Entry stored in json file, 

for idx in range(20):
    idx_name = f'{idx}: User Entry'
    if idx_name in average_formulas:
        first_entry_idx = average_formulas.index(idx_name)
        break
        
# pick the first User Entry and the stored values for each development period (column)
user_entry_values = [aDFM.AverageRatioValues(j, first_entry_idx+1) for j in org_rng]

# find the selected average formula index
[aDFM.SelectedRatios(DevIndex=j) for j in dev_rng]

[9, 5, 6, 6, 6, 6, 6, 9, 9, 10]

In [ ]:
aDFM.SelectedEstimateValues(10)   # The selected cell (green fill in UI)

1.0005

In [ ]:
aDFM.SelectedRatioValues(3)  # the first row of blue cell ??

1.171904542227624

In [20]:
aDFM.TailLabel

'113 - Ult'

In [ ]:
[aDFM.TailReserves(i) for i in range(1, 11)]

[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]

In [21]:
aDFM.TailFactorValue

<bound method TailFactorValue of <COMObject Item>>

In [22]:
[aDFM.TailFactorValue(i) for i in range(1, 11)]

[1.0005,
 1.000137828788868,
 1.0006987584896967,
 1.0002200974639046,
 1.0000007569624811,
 1.0,
 1.0,
 1.0017307651269096,
 1.0,
 1.0]

In [20]:
basis_value = [aDFM.SummaryRatioBasis.ValuesByIndex(i) for i in range(1, aDFM.OriginCount+1)]
basis_value

[118545.31391118311,
 120989.35186750931,
 124997.6300462914,
 126444.12385311208,
 119730.18450113725,
 129092.40111071248,
 116535.40306864819,
 122583.97432000918,
 107245.49944350947,
 116350.46495999147,
 115311.419726075,
 112043.14908647531,
 113301.82210771213,
 116319.77630576612,
 119092.58959421536,
 130249.82649558535,
 111537.41318818337,
 133496.70294472712,
 125066.66242523042,
 125719.4147326109,
 115862.01116678398,
 120231.3872499467,
 118926.82231741707,
 126457.88880560976,
 112516.08396747206,
 125544.92874994934,
 124899.34918338027,
 116607.2625173365,
 117088.67683028904,
 119681.4884021856,
 113051.40216706292,
 101225.73323023421,
 106047.89324724014,
 103390.55964483373,
 103922.68664805642,
 103857.29473935415,
 103491.40784698403,
 108485.28805621449,
 106785.80471407507,
 110352.78523834488]

In [ ]:
aDFM.OutputVector.Modified

In [7]:
?aDFM.SetUserRatios

Signature:
aDFM.SetUserRatios(
    DevIndex=<PyOleMissing object at 0x000001868B4C98C0>,
    AvgIndex=<PyOleMissing object at 0x000001868B4C98C0>,
    arg2=<PyOleMissing object at 0x000001868B4C98C0>,
)
Docstring: <no docstring>
File:      e:\xwspace\repos\arcrho\python-api\migration\references\<comobject item>
Type:      method

In [8]:
?aDFM.SetSelectedRatios

Signature:
aDFM.SetSelectedRatios(
    DevIndex=<PyOleMissing object at 0x000001868B4C98C0>,
    arg1=<PyOleMissing object at 0x000001868B4C98C0>,
)
Docstring: <no docstring>
File:      e:\xwspace\repos\arcrho\python-api\migration\references\<comobject item>
Type:      method

In [ ]:
final_value = 8.8888  # growth adjustments applied
# Normally, user entry is the 10th row
aDFM.SetUserRatios(1, 10, final_value)
aDFM.SetSelectedRatios(1, 10)
aDFM.save()

In [ ]:
aDFM.Ratios(OriginIndex=1, DevIndex=1)

In [ ]:
[aDFM.OriginLabel(i) for i in dev_rng]

In [ ]:
[aDFM.DevelopmentLabel(i) for i in dev_rng]

In [ ]:
[aDFM.Ultimates(i) for i in org_rng]

In [ ]:
aDFM.CellNotes

In [ ]:
print(aDFM.CellNotes)
# Tab, X Label, Y Label, User, Date, Notes  -- The labels in ResQ UI
# The value for X Label shows 2-14 in aDFM.CellNotes means '(1) 2-14' from aDFM.DevelopmentLabel(1)

## Write DFM data into ResQ

In [ ]:
aDFM.SetExcludedRatios(OriginIndex=1, DevIndex=1, arg2=1)  # set cell (1, 1) as excluded
aDFM.SetExcludedRatios(OriginIndex=1, DevIndex=1, arg2=0)  # set cell (1, 1) as included (not excluded)

aDFM.SetSelectedRatios(DevIndex=1, arg1=2)  # set the selected average formula index to be 2 (2nd formula) for development column 1  

aDFM.Notes = 'New Notes'  # need to use /r/n to change line instead of /n

aDFM.Save()  # Save all changes to real Database

# Bornhuetter Ferguson (BF Method)

Use this section to inspect the BF method properties currently needed by ArcRho.

## Read properties

In [7]:
BF_MethodName = r'C 41 - BF Reported ex CWOP'

bf_method = reserving_class.BFMethods().Item(BF_MethodName)
bf_method.Name

'C 41 - BF Reported ex CWOP'

In [8]:
bf_method.OriginLabel(OriginIndex=1)

'2017'

In [11]:
bf_method.PercentageDevelopedValues(10)

0.4039833077803641

In [33]:
PERCENTAGE_DEVELOPED_TYPES = {
    0: "Latest/Ultimates",
    1: "Pattern Vector",
    2: "DFM dev factors",
    3: "DFM dev factors (adj)",
}

PRIOR_TYPES = {
    0: "Ultimates",
}


def read_resq_path(obj, property_path):
    value = obj
    for property_name in property_path.split("."):
        value = getattr(value, property_name)
    return value


def describe_bf_value(property_path, value):
    if property_path == "PercentageDevelopedType" and isinstance(value, int):
        return f"{value}: {PERCENTAGE_DEVELOPED_TYPES.get(value, 'Unknown')}"

    if property_path == "PriorType" and isinstance(value, int):
        return f"{value}: {PRIOR_TYPES.get(value, 'Unknown')}"

    return value


def print_bf_properties(bf_method):
    property_paths = [
        "Name",
        "OutputVector.DatasetType.Name",
        "OriginLength",
        "PercentageDeveloped.Name",
        "PercentageDevelopedType",
        "Latest.Name",
        "Prior.Name",
        "PriorType",
    ]

    rows = []
    for property_path in property_paths:
        value = read_resq_path(bf_method, property_path)
        rows.append((property_path, describe_bf_value(property_path, value)))

    width = max(len(property_path) for property_path, _ in rows)
    for property_path, value in rows:
        print(f"{property_path:<{width}}  {value}")

    return rows


bf_properties = print_bf_properties(bf_method)

Name                           C 41 - BF Reported ex CWOP
OutputVector.DatasetType.Name  C 41 - BF Reported ex CWOP
OriginLength                   12
PercentageDeveloped.Name       C 42 - Reported ex CWOP DFM w/ Selected LDFs  
PercentageDevelopedType        0: Latest/Ultimates
Latest.Name                    Claim Counts--Reported ex CWOP
Prior.Name                     C 42a - Prior for BF Reported ex CWOP
PriorType                      0: Ultimates


# Berquist Sherman

In [30]:
# Berquist Sherman (bs)
# Settlement Rate (sr)
bs_sr_name = 'Gross Loss--Paid  - B&S Settlement Rate Adjustment' # bs triangle (with method attached) instance name
bs_method = reserving_class.GetBerquistShermanSR(bs_sr_name)

In [31]:
bs_method.OutputTriangle.Status

0

In [ ]:
bs_method.OutputTriangle.MethodType

In [ ]:
# "Paid Loss" (Input Triangle 1)
bs_method.PaidClaims.Name

# "Closed Claim Counts" (Input Triangle 2)
bs_method.ClosedClaimNos.Name

# "Ultimate Claim Counts" (Input Vector)
bs_method.UltimateClaimNos.Name

# Output Triangle Dataset Type
bs_method.OutputTriangle.DatasetType.Name

bs_method.OriginLength # int
bs_method.DevelopmentLength # int

In [ ]:
output_tri = bs_method.OutputTriangle

In [ ]:
output_tri.ValuesByIndex(1,1)

# Result Selection

In [3]:
ResultSelectionName = 'C 91 -  Current Qtr Indicated'
ResultSelectionName = 'C 92 -  Current Qtr Selected'
result_selection = reserving_class.GetResultSelection(ResultSelectionName)

# Result Selection output vectors use MethodType 4.
result_selection.OutputVector.Name
result_selection.OutputVector.DatasetType.Name
result_selection.OutputVector.MethodType


4

In [4]:
result_selection.OutputVector.Status

2

In [19]:
[result_selection.UltimateOverridden(OriginIndex=i) for i in range(1, result_selection.OriginCount+1)]

[False, False, False, False, False, False, False, False, True, False]

In [ ]:
[result_selection.RatioBasisValues(OriginIndex=i, OriginLength=result_selection.OriginLength) for i in range(1, result_selection.OriginCount+1)]


[495900.4586265378,
 485821.1239613796,
 448192.84748624347,
 486174.43903027766,
 460779.87338736065,
 475678.4297142625,
 487006.8372017533,
 467744.0933191544,
 427747.1807307569,
 260782.13249528164]

In [10]:
result_selection.OriginCount

10

In [ ]:
result_selection.OriginLength

In [ ]:
result_selection.OutputVector.PeriodLength

In [ ]:
# Basic shape and labels
result_selection.OriginLength
result_selection.OriginCount
result_selection.DatasetCount
result_selection.OriginLabel(1)


In [ ]:
result_selection.RatioBasisDataset(1).Name

In [ ]:
# Source datasets used by the Result Selection method. Dataset(i) is 1-based.
source_index = 1
source_dataset = result_selection.Dataset(source_index)

source_dataset.Name
source_dataset.DatasetType.Name
source_dataset.DatasetType.DataFormat  # 0=Triangle, 1=Vector


In [ ]:
# Read selected source values, weights, and selected ultimate by origin row.
origin_index = 1
origin_length = result_selection.OriginLength

source_value = result_selection.DatasetValues(source_index, origin_index, origin_length)
source_weight = result_selection.Weights(source_index, origin_index)
selected_ultimate = result_selection.Ultimates(origin_index, origin_length)

source_value, source_weight, selected_ultimate


In [ ]:
# Build a compact table of source values and weights for all origins.
rows = []
for row_index in range(1, result_selection.OriginCount + 1):
    row = {
        "origin": result_selection.OriginLabel(row_index),
        "selected_ultimate": result_selection.Ultimates(row_index, result_selection.OriginLength),
    }
    for dataset_index in range(1, result_selection.DatasetCount + 1):
        dataset_name = result_selection.Dataset(dataset_index).Name
        row[f"{dataset_name} value"] = result_selection.DatasetValues(dataset_index, row_index, result_selection.OriginLength)
        row[f"{dataset_name} weight"] = result_selection.Weights(dataset_index, row_index)
    rows.append(row)

rows[:3]


In [ ]:
# Write one Result Selection weight back to ResQ. Use with care on a scratch/test project.
# result_selection.SetWeights(source_index, origin_index, source_weight)
# result_selection.Save()


# Cleanup

In [12]:
t1 = reserving_class.GetTriangle('Claim Counts--CWP')
mtime = t1.Modified
mdate = f"{mtime.year}-{mtime.month}-{mtime.day}"
mdate

'2026-9-1'

In [ ]:
project.Vectors

In [ ]:
for i in project.Triangles():
    mtime = i.Modified
    mdate = f"{mtime.year}-{mtime.month}-{mtime.day}"
    if mdate != '2026-9-1':
        print(i.Path)
        print(i.Name)